# Topological feature selection: worked example

We score the variables of a multivariate time series by their contribution to the
topology of its sliding-window embedding, using projected stochastic subgradient
descent on the persistence diagram.

The data consist of three cosines sharing a common period with different amounts of Gaussian noise. The method should identify the least-noisy signal as the dominant contributor to the periodic structure.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np

from src.data_utils import generate_three_sine_dataset, sw_dist_mats, weighted_matrix_sum
from src.persistence_optimization import (
    persistence,
    estimate_grad_norm,
    stochastic_subgradient_descent,
)
from src.plotting_utils import (
    plot_time_series,
    plot_diagram,
    plot_simplex_path,
    plot_objective,
)

## Data

Three cosines of period 50 with noise standard deviations 1, 2, and 3. The faint
curves show the underlying noise-free signals.

In [ ]:
dataset, clean = generate_three_sine_dataset(
    n_points=300, period=50, noise_sds=(1, 2, 3), seed=6, return_clean=True
)

plot_time_series(
    dataset, y_bounds=(-5, 5), marker_style='.', marker_size=5,
    underlay=clean,
)

## Sliding-window embedding

With a window length of 250, each signal's sliding-window embedding traces a loop
whose persistence reflects how cleanly periodic the signal is. We form one
distance matrix per signal; the optimizer searches over convex combinations of
these matrices.

In [ ]:
WINDOW = 250
dist_mats = sw_dist_mats(dataset, window=WINDOW)          # one matrix per signal
glob_dist = sw_dist_mats(dataset, window=WINDOW, glob=True)

plot_diagram(persistence(glob_dist))

## Optimization

We maximize the maximum persistence of the degree-1 diagram.
The hyperparameters are set relative to the subgradient norm at the barycenter:
`eta_0` fixes the initial step length, `d` halves it over the run, and `sigma`
sets the gradient noise to one tenth of the initial gradient.

In [ ]:
STEP_SIZE = (2 ** -4) / 3       # initial step length
N_STEPS   = 500

norm = estimate_grad_norm(dist_mats, num_of_features=1)

path = stochastic_subgradient_descent(
    dist_mats,
    num_of_steps=N_STEPS,
    num_of_features=1,
    eta_0=STEP_SIZE / norm,
    d=1.0 / N_STEPS,
    sigma=0.1 * norm,
    col_names=['1', '2', '3'],
    seed=0,
)

print('Final score:', path.iloc[-1].round(3).to_dict())

## Results

The path through the 2-simplex starts at the barycenter and converges to the
vertex of the least-noisy signal.

In [ ]:
plot_simplex_path(path, vertex_labels=['1', '2', '3'])

In [ ]:
plot_objective([path], dist_mats, num_of_features=1)

In [ ]:
plot_diagram(persistence(weighted_matrix_sum(dist_mats, path.iloc[-1].values)))